# Create PCHRD Awards from Official Ongoing Projects API

Creates OpenAlex award rows for Philippine Council for Health Research and Development (DOST-PCHRD) ongoing funded health research projects.

**Awarding body:** Philippine Council for Health Research and Development - `F4320335609` (PH; no DOI/ROR in the OpenAlex public API result).

**Source:** official PCHRD `Funded Projects -> Ongoing Projects` WordPress REST API at `https://www.pchrd.dost.gov.ph/wp-json/wp/v2/ongoing_projects`, plus official `projects_category` and `implementing_agency` taxonomy endpoints.

**Schema choices:**
- One row per official ongoing project. `funder_award_id` = `pchrd-{wp_post_id}`.
- `funder_scheme` = official project category taxonomy term.
- `funding_type = 'research'`.
- PCHRD does not publish per-project budgets, PI names, or project-start dates in this API. `amount`/`currency` are NULL by section 6.7 waiver. `start_date` uses the official WordPress project-record publication date as a source-observed date and is documented as such in validation.
- `lead_investigator` is organization-shaped when a clean implementing agency is published: given/family/orcid NULL, `affiliation.name = source_implementing_agency`, `affiliation.country = 'PH'`.

**S3 location:** `s3a://openalex-ingest/awards/pchrd/pchrd_ongoing_projects.parquet`

**Local full-source validation on 2026-05-28:** 185 rows; 0 duplicate `funder_award_id`s; 100.0% title/category/source-posted-date/landing-page coverage; 99.5% implementing-agency coverage; all source-posted years 2022; amount/currency NULL by section 6.7 waiver.

Contractor has no S3/Databricks access; admin must upload the parquet, run this notebook, inspect verification cells, and decide later whether a job YAML is warranted.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.pchrd_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/pchrd/pchrd_ongoing_projects.parquet`;


In [ ]:
%sql
-- Local full-source validation on 2026-05-28 found 185 official ongoing projects.
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.pchrd_awards_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.pchrd_awards_raw;


In [ ]:
%sql
SELECT
    funder_award_id,
    display_name,
    project_category,
    source_implementing_agency,
    source_posted_date,
    landing_page_url,
    amount,
    currency
FROM openalex.awards.pchrd_awards_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'pchrd_awards_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency';


In [ ]:
%sql
-- Native key uniqueness: must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.pchrd_awards_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC, funder_award_id
LIMIT 20;


In [ ]:
%sql
-- Core coverage check. Amount/currency are expected NULL by section 6.7 waiver.
SELECT
    COUNT(*) AS rows,
    COUNT(display_name) AS has_title,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    COUNT(project_category) AS has_project_category,
    ROUND(try_divide(COUNT(project_category), COUNT(*)) * 100.0, 1) AS pct_project_category,
    COUNT(source_implementing_agency) AS has_implementing_agency,
    ROUND(try_divide(COUNT(source_implementing_agency), COUNT(*)) * 100.0, 1) AS pct_implementing_agency,
    COUNT(source_posted_date) AS has_source_posted_date,
    ROUND(try_divide(COUNT(source_posted_date), COUNT(*)) * 100.0, 1) AS pct_source_posted_date,
    COUNT(landing_page_url) AS has_landing_page_url,
    ROUND(try_divide(COUNT(landing_page_url), COUNT(*)) * 100.0, 1) AS pct_landing_page_url,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS has_currency,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_currency
FROM openalex.awards.pchrd_awards_raw;


In [ ]:
%sql
-- Amount/currency are NULL because the official PCHRD API does not publish budgets.
SELECT
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount,
    COUNT(currency) AS rows_with_currency
FROM openalex.awards.pchrd_awards_raw;


In [ ]:
%sql
-- Source-observed date shape. These are WordPress record publication dates, not asserted project-start dates.
SELECT
    MIN(TRY_TO_DATE(source_posted_date, 'yyyy-MM-dd')) AS min_source_posted_date,
    MAX(TRY_TO_DATE(source_posted_date, 'yyyy-MM-dd')) AS max_source_posted_date,
    COUNT(DISTINCT source_year) AS distinct_source_years
FROM openalex.awards.pchrd_awards_raw;


In [ ]:
%sql
SELECT project_category, COUNT(*) AS rows
FROM openalex.awards.pchrd_awards_raw
GROUP BY project_category
ORDER BY rows DESC, project_category;


In [ ]:
%sql
SELECT source_implementing_agency, COUNT(*) AS rows
FROM openalex.awards.pchrd_awards_raw
GROUP BY source_implementing_agency
ORDER BY rows DESC, source_implementing_agency
LIMIT 25;


## Step 1.6: Funder Existence Assertion

In [ ]:
%sql
-- Public OpenAlex API pre-flight on 2026-05-28 verified F4320335609:
-- Philippine Council for Health Research and Development; country PH; no DOI/ROR.
WITH expected AS (
    SELECT
        4320335609 AS expected_funder_id,
        'Philippine Council for Health Research and Development' AS expected_display_name
),
raw_funders AS (
    SELECT
        COUNT(DISTINCT TRY_CAST(funder_id AS BIGINT)) AS distinct_raw_funder_ids,
        MIN(TRY_CAST(funder_id AS BIGINT)) AS min_raw_funder_id,
        MAX(TRY_CAST(funder_id AS BIGINT)) AS max_raw_funder_id
    FROM openalex.awards.pchrd_awards_raw
)
SELECT
    CASE
        WHEN r.distinct_raw_funder_ids = 1
         AND r.min_raw_funder_id = e.expected_funder_id
         AND r.max_raw_funder_id = e.expected_funder_id
        THEN 'OK'
        ELSE raise_error(CONCAT(
            'PCHRD raw funder_id mismatch: distinct=', r.distinct_raw_funder_ids,
            ', min=', r.min_raw_funder_id,
            ', max=', r.max_raw_funder_id,
            ', expected=', e.expected_funder_id
        ))
    END AS funder_assertion,
    e.expected_funder_id AS funder_id,
    e.expected_display_name AS display_name,
    'https://openalex.org/F4320335609' AS openalex_funder_url
FROM expected e
CROSS JOIN raw_funders r;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.pchrd_awards
USING delta
AS
WITH normalized AS (
    SELECT
        *,
        TRY_CAST(funder_id AS BIGINT) AS funder_id_bigint,
        TRY_TO_DATE(source_posted_date, 'yyyy-MM-dd') AS source_posted_date_parsed,
        NULLIF(TRIM(source_implementing_agency), '') AS implementing_agency_clean,
        NULLIF(TRIM(project_category), '') AS project_category_clean,
        NULLIF(TRIM(description), '') AS description_clean
    FROM openalex.awards.pchrd_awards_raw
    WHERE funder_award_id IS NOT NULL
      AND display_name IS NOT NULL
)
SELECT
    abs(xxhash64(CONCAT(TRY_CAST(funder_id_bigint AS STRING), ':', LOWER(funder_award_id)))) % 9000000000 AS id,
    display_name,
    description_clean AS description,
    funder_id_bigint AS funder_id,
    funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(funder_id_bigint AS STRING)) AS id,
        'Philippine Council for Health Research and Development' AS display_name,
        CAST(NULL AS STRING) AS ror_id,
        CAST(NULL AS STRING) AS doi
    ) AS funder,
    'research' AS funding_type,
    project_category_clean AS funder_scheme,
    'pchrd_ongoing_projects' AS provenance,
    source_posted_date_parsed AS start_date,
    CAST(NULL AS DATE) AS end_date,
    YEAR(source_posted_date_parsed) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN implementing_agency_clean IS NULL THEN NULL
        ELSE struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            source_posted_date_parsed AS role_start,
            struct(
                implementing_agency_clean AS name,
                'PH' AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(funder_id_bigint AS STRING), ':', LOWER(funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM normalized;


In [ ]:
%sql
SELECT COUNT(*) AS transformed_rows
FROM openalex.awards.pchrd_awards;


In [ ]:
%sql
-- Confirm generated OpenAlex award IDs are unique.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.pchrd_awards
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY rows DESC, id
LIMIT 20;


## Step 3: Delete Existing Priority Slice and Insert

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pchrd_ongoing_projects' AND priority = 162;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    162 AS priority
FROM openalex.awards.pchrd_awards;


## Step 6: Verification

In [ ]:
%sql
SELECT
    'raw' AS table_name,
    COUNT(*) AS rows
FROM openalex.awards.pchrd_awards_raw
UNION ALL
SELECT
    'transformed' AS table_name,
    COUNT(*) AS rows
FROM openalex.awards.pchrd_awards
UNION ALL
SELECT
    'openalex_awards_raw_priority_162' AS table_name,
    COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pchrd_ongoing_projects'
  AND priority = 162;


In [ ]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    MIN(start_year) AS min_source_observed_year,
    MAX(start_year) AS max_source_observed_year,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(lead_investigator.affiliation.name) AS rows_with_implementing_agency
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pchrd_ongoing_projects'
  AND priority = 162;


In [ ]:
%sql
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    COUNT(lead_investigator.affiliation.name) AS rows_with_implementing_agency
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pchrd_ongoing_projects'
  AND priority = 162
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme;


In [ ]:
%sql
SELECT
    id,
    display_name,
    funder_scheme,
    start_date AS source_observed_date,
    lead_investigator.affiliation.name AS implementing_agency,
    landing_page_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pchrd_ongoing_projects'
  AND priority = 162
ORDER BY display_name
LIMIT 25;


In [ ]:
%sql
-- Dedup safety inside this provenance/priority slice: must return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pchrd_ongoing_projects'
  AND priority = 162
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY rows DESC, id
LIMIT 20;



## Handoff Notes

Contractor-complete state stops here. Admin must:

- run `scripts/local/pchrd_to_s3.py --allow-shrink` only after reviewing any future shrink, otherwise run without override;
- upload the parquet to `s3a://openalex-ingest/awards/pchrd/pchrd_ongoing_projects.parquet`;
- run this notebook on Databricks;
- inspect the Step 6 verification cells, especially the documented section 6.7 amount waiver and source-observed-date note;
- only then decide whether to add scheduled job YAML.
